In [1]:
uri = 'https://api.football-data.org/v4/competitions/WC/matches'
headers = {'X-Auth-Token': 'b006736168e34387975ae15e83b341a4'}

In [2]:
import requests
import json
import sys
import os
import streamlit as st
import numpy as np
from names_eng_to_nor import ENGLISH_TO_NORWEGIAN

In [3]:
response = requests.get(uri, headers=headers, timeout=10)
response.raise_for_status()

In [4]:
np.sum([1,2,3,4,5])

np.int64(15)

In [5]:
matches = []
knockout_stages = {
    'round_of_32': [],
    'round_of_16': [],
    'quarter_finals': [],
    'semi_finals': [],
    'finals_teams': [],
    'finals_winner': None
}

# Stage mapping from API to our format
stage_mapping = {
    'LAST_32': 'round_of_32',
    'LAST_16': 'round_of_16',
    'QUARTER_FINALS': 'quarter_finals',
    'SEMI_FINALS': 'semi_finals',
    'FINAL': 'finals'
}

# Parse JSON response
for match in response.json()['matches']:
    home_team = match['homeTeam']['name']
    away_team = match['awayTeam']['name']
    stage = match.get('stage', 'GROUP_STAGE')
    status = match.get('status')

    # Convert team names to Norwegian
    home_team_nor = ENGLISH_TO_NORWEGIAN.get(home_team, home_team)
    away_team_nor = ENGLISH_TO_NORWEGIAN.get(away_team, away_team)

    if stage == 'GROUP_STAGE' and status in ['FINISHED', 'IN_PLAY', 'TIMED']:
        home_score = match['score']['fullTime']['home']
        away_score = match['score']['fullTime']['away']
        matches.append({
            'home_team_eng': home_team,
            'away_team_eng': away_team,
            'home_team': home_team_nor,
            'away_team': away_team_nor,
            'home_score': home_score,
            'away_score': away_score,
            'score_str': f"{home_score}–{away_score}" if home_score is not None else "–"
        })
    elif stage in stage_mapping:
        # Add both participating teams to their stage list regardless of match status,
        # so the list reflects all scheduled/played teams, not just winners.
        if stage == 'FINAL':
            if home_team_nor not in knockout_stages['finals_teams']:
                knockout_stages['finals_teams'].append(home_team_nor)
            if away_team_nor not in knockout_stages['finals_teams']:
                knockout_stages['finals_teams'].append(away_team_nor)
            if status == 'FINISHED':
                home_score = match['score']['fullTime']['home']
                away_score = match['score']['fullTime']['away']
                if home_score > away_score:
                    knockout_stages['finals_winner'] = home_team_nor
                elif away_score > home_score:
                    knockout_stages['finals_winner'] = away_team_nor
        else:
            if stage == 'LAST_32':
                print(match)

            stage_key = stage_mapping[stage]
            if home_team_nor not in knockout_stages[stage_key]:
                knockout_stages[stage_key].append(home_team_nor)
            if away_team_nor not in knockout_stages[stage_key]:
                knockout_stages[stage_key].append(away_team_nor)

{'area': {'id': 2267, 'name': 'World', 'code': 'INT', 'flag': None}, 'competition': {'id': 2000, 'name': 'FIFA World Cup', 'code': 'WC', 'type': 'CUP', 'emblem': 'https://crests.football-data.org/wm26.png'}, 'season': {'id': 2398, 'startDate': '2026-06-11', 'endDate': '2026-07-19', 'currentMatchday': 3, 'winner': None}, 'id': 537417, 'utcDate': '2026-06-28T19:00:00Z', 'status': 'FINISHED', 'matchday': None, 'stage': 'LAST_32', 'group': None, 'lastUpdated': '2026-06-29T05:20:12Z', 'homeTeam': {'id': 774, 'name': 'South Africa', 'shortName': 'South Africa', 'tla': 'RSA', 'crest': 'https://crests.football-data.org/9396.svg'}, 'awayTeam': {'id': 828, 'name': 'Canada', 'shortName': 'Canada', 'tla': 'CAN', 'crest': 'https://crests.football-data.org/canada.svg'}, 'score': {'winner': 'AWAY_TEAM', 'duration': 'REGULAR', 'fullTime': {'home': 0, 'away': 1}, 'halfTime': {'home': 0, 'away': 0}}, 'odds': {'msg': 'Activate Odds-Package in User-Panel to retrieve odds.'}, 'referees': [{'id': 38806, 'na

In [12]:
print("Matches fetched and processed successfully.")

Matches fetched and processed successfully.


In [13]:
display(knockout_stages)

{'round_of_32': ['Sør-Afrika',
  'Canada',
  'Brasil',
  'Japan',
  'Tyskland',
  None,
  'Nederland',
  'Marokko',
  'Elfenbenskysten',
  'Mexico',
  'USA',
  'Bosnia-Hercegovina',
  'Sveits',
  'Australia',
  'Argentina'],
 'round_of_16': [None],
 'quarter_finals': [None],
 'semi_finals': [None],
 'finals_teams': [None],
 'finals_winner': None}

In [16]:
print(response.json()['matches'][73]['stage'])

LAST_32


In [18]:
response.json()['matches'][74]

{'area': {'id': 2267, 'name': 'World', 'code': 'INT', 'flag': None},
 'competition': {'id': 2000,
  'name': 'FIFA World Cup',
  'code': 'WC',
  'type': 'CUP',
  'emblem': 'https://crests.football-data.org/wm26.png'},
 'season': {'id': 2398,
  'startDate': '2026-06-11',
  'endDate': '2026-07-19',
  'currentMatchday': 2,
  'winner': None},
 'id': 537415,
 'utcDate': '2026-06-29T20:30:00Z',
 'status': 'TIMED',
 'matchday': None,
 'stage': 'LAST_32',
 'group': None,
 'lastUpdated': '2026-06-22T18:05:47Z',
 'homeTeam': {'id': 759,
  'name': 'Germany',
  'shortName': 'Germany',
  'tla': 'GER',
  'crest': 'https://crests.football-data.org/759.svg'},
 'awayTeam': {'id': None,
  'name': None,
  'shortName': None,
  'tla': None,
  'crest': None},
 'score': {'winner': None,
  'duration': 'REGULAR',
  'fullTime': {'home': None, 'away': None},
  'halfTime': {'home': None, 'away': None}},
 'odds': {'msg': 'Activate Odds-Package in User-Panel to retrieve odds.'},
 'referees': []}